# Modeling Noise

This notebook demonstrates the use of the `NoiseModel` and `Channel` objects to create and use noise models.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import quax as qx

from pyquil.noise import Channel, MeasurementChannel, NoiseModel

from pyquil.gates import CZ, MEASURE, RX
from pyquil.quilbase import Gate

## Modeling gate noise with `Channels`

### Average Fidelity vs Pauli Fidelity

There are two common metrics of gate fidelity. The _Average Gate Fidelity_ ($F$) and the _Pauli Gate Fidelity_ ($\chi_{00}$). 

The average gate fidelity is not stable under tensor products

$$ F_{1} \otimes F_{2} \neq F_{1}F_{2}$$

$$ \chi_{1} \otimes \chi_{2} = \chi_{1}\chi_{2}$$

#### Average Gate Fidelity
Average Gate Fidelity is defined as:

$$ F = \int{d\psi \langle \psi | \mathcal{E} (| \psi \rangle \langle \psi |) | \psi \rangle} $$

or average gate _infidelity_,

$$ r = 1 - \int{d\psi \text{Tr}(\psi, \mathcal{E}(\psi))} $$

#### Process or Pauli Fidelity

While the Pauli fidelity ($\chi_{00}$) is defined as 


$$ \chi_{00} = F(\mathcal{E}) = \text{Tr}(\mathcal{E})/d^2$$

or the infidelity,

$$ e_{F}  = 1 - \text{Tr}(\mathcal{E})/d^2$$

#### Converting 

The average gate fidelity can be related to the Pauli fidelity by:

$$ \chi_{00} = \frac{(d+1)F - 1}{d}$$

and the Pauli fidelity to the average gate fidelity by:

F¯=dχ00+1d+1

$$F = \frac{d \chi_{00} + 1}{d+1}$$

Note that $d$ is the dimension of the Hilbert space, whcih for qubits is $2^n$.

$r$ is the average gate infidelity and $e_F$ is the process infidelity.

$$ e_{F} = (1+1/d)r $$

##### 1Q Gates
For 1-qubit gates, this means the error rates can be related by simple rules of thumb:

$$ r = \frac{2}{3}e_{F}$$

$$ e_{F} = \frac{3}{2}r$$


##### 2Q Gates
For 2-qubit gates, the rules are,

$$ r = \frac{4}{5}e_{F}$$

$$ e_{F} = \frac{5}{4}r$$

[1] A. Carignan-Dugas, J. J. Wallman, and J. Emerson, “Bounding the average gate fidelity of composite channels using the unitarity,” New J. Phys., vol. 21, no. 5, p. 053016, May 2019, doi: 10.1088/1367-2630/ab1800.


### Depolarizing Channels

Depolarizing channels apply all error with equal probability, leading towards a mixed state.

In [ ]:
# create a single-qubit depolarizing channel based on the reported gate infidelity
fidelity = 0.995
channel = Channel.from_gate_fidelity(inst=RX(jnp.pi / 2, 0), fidelity=fidelity)
print(channel)
print(f"Average Gate Fidelity: {100 * channel.fidelity:.2f}%")
print(f"Pauli Gate Fidelity: {100 * channel.pauli_fidelity:.2f}%")

channel.plot()

In [ ]:
# create a 2q depolarizing channel
fidelity = 0.995
channel = Channel.from_gate_fidelity(inst=CZ(0, 1), fidelity=fidelity)

print(f"Average Gate Fidelity: {100 * channel.fidelity:.2f}%")
print(f"Pauli Gate Fidelity: {100 * channel.pauli_fidelity:.2f}%")

channel.plot()

### Decoherence channel

The decoherence channel models the effective amplitude damping (T1) and dephasing (T2) during the gate.

In [ ]:
# create a single-qubit depolarizing channel based on the reported gate infidelity
fidelity = 0.995
channel = Channel.from_coherence_times(inst=RX(jnp.pi / 2, 0), gate_duration=24e-9, t1s=[20e-6])

print(f"Average Gate Fidelity: {100 * channel.fidelity:.2f}%")

channel.plot()

In [ ]:
# create a single-qubit depolarizing channel based on the reported gate infidelity
fidelity = 0.995
channel = Channel.from_coherence_times(inst=CZ(0, 1), gate_duration=100e-9, t1s=[20e-6, 30e-6], t2s=[25e-6, 20e-6])

print(f"Average Gate Fidelity: {100 * channel.fidelity:.2f}%")

channel.plot()

### Mixture channels

Mixture channel apply unitaries stochastically, after the gate.

In [ ]:
# create a single-qubit depolarizing channel based on the reported gate infidelity
fidelity = 0.995
channel = Channel.from_mixture(inst=RX(jnp.pi, 0), constituents=[qx.gates.RX(jnp.pi / 12)], probabilities=[0.1])

print(f"Average Gate Fidelity: {100 * channel.fidelity:.2f}%")

channel.plot()

In [ ]:
# create a single-qubit depolarizing channel based on the reported gate infidelity
fidelity = 0.995
channel = Channel.from_mixture(
    inst=CZ(0, 1), constituents=[qx.gates.RXX(jnp.pi / 12), qx.gates.X | qx.gates.Z], probabilities=[0.1, 0.02]
)

print(f"Average Gate Fidelity: {100 * channel.fidelity:.2f}%")

channel.plot()

### Composing channels

Noise channels can be composed using the `@` or the `__matmul__` operator. For example, we can compose a depolarizing and decoherence channel. Note that channels must be defined on the same qubits and the same gate to be composed.

In [ ]:
fidelity = 0.995
decoherence_channel = Channel.from_coherence_times(inst=RX(jnp.pi, 0), gate_duration=120e-9, t1s=[10e-6])
depolarizing_channel = Channel.from_gate_fidelity(inst=RX(jnp.pi, 0), fidelity=fidelity)
print(f"Decoherence Average Gate Fidelity: {100 * decoherence_channel.fidelity:.2f}%")
print(f"Depolarizing Average Gate Fidelity: {100 * depolarizing_channel.fidelity:.2f}%")

composed_channel = decoherence_channel @ depolarizing_channel
print(f"Composed Average Gate Fidelity: {100 * composed_channel.fidelity:.2f}%")

channel.plot()

### Tensor product of channels

Computing the tensor product of channels should result in a `CycleChannel`

In [ ]:
decoherence_channel = Channel.from_coherence_times(inst=RX(jnp.pi, 0), gate_duration=120e-9, t1s=[10e-6])
depolarizing_channel = Channel.from_gate_fidelity(inst=RX(jnp.pi, 1), fidelity=fidelity)

tensor_channel = decoherence_channel | depolarizing_channel
print(tensor_channel)

### Coherent errors

We can also construct a channel with coherent errors.

In [ ]:
process = qx.gates.RX(jnp.pi / 2 + jnp.pi / 12)  # here we add a little extra rotation
channel = Channel(inst=RX(jnp.pi / 2, 0), process=qx.to_superop(process))

channel.plot()

Coherent errors can be more complicated than over-rotations. Here we add an off-axis rotation, a typical result of a detuning error.

Rather than using a gate to construct this error, we exponentiate a Hamiltonian:

The standard `RX(pi/2)` can be written

$e^{-i\frac{\pi}{4} \text{X}}$

To add a small detuning, we use:

$e^{-i[\frac{\pi}{4} \text{X} + \frac{\pi}{40} \text{Z}]}$

In [ ]:
phi = jnp.pi / 2
process = qx.cis((qx.gates.X + 0.1 * qx.gates.Z) * (-0.5 * phi))
channel = Channel(inst=RX(jnp.pi / 2, 0), process=qx.to_superop(process))

channel.plot()

### Modeling crosstalk

We can use the same trick as above to construct a channel with crosstalk. First, we'll construct a tensor product of two RX(pi/2) gates.

$$e^{-i\frac{\pi}{4} [\text{XI} + \text{IX}]}$$



In [ ]:
phi = jnp.pi / 2
process = qx.cis(
    ((qx.gates.X | qx.gates.I) + (qx.gates.I | qx.gates.X) + 0.1 * (qx.gates.Z | qx.gates.Z)) * (-0.5 * phi)
)


channel = Channel(
    inst=Gate("SXxSX", [], [0, 1]),
    process=qx.to_superop(process),
    target_unitary=qx.gates.RX(jnp.pi / 2) | qx.gates.RX(jnp.pi / 2),
)
channel.plot()

### Twirling noise channels

We can Pauli twirl a  noise channel.

In [ ]:
channel = Channel.from_random_coherent_error(inst=RX(jnp.pi, 0), process_fidelity=0.95)
channel.plot().show()

twirled_channel = channel.pauli_twirl()
twirled_channel.plot().show()

### Leakage noise channnel

We can model leakage by adding leakage and seepage operators.

In [ ]:
superop = (
    qx.stochastic_leakage_operators(0.02) @ qx.depolarizing_channel_superoperator(0.05, (2,)) @ qx.gates.RX(jnp.pi)
)
channel = Channel(RX(jnp.pi, 0), superop)

channel.plot()

## Modeling Measurements with `QuantumInstruments`

### Standard measurement

Here we have a fairly pedestrian 95% accurate measurement with no QND error and symmetric errors.

In the primary plot, we see that the measurement is a perfect projector to $|0\rangle \langle 0|$ and  $|1\rangle \langle 1|$.

The the outcome plots, we see faint elements in the lower right and upper left for 0 and 1 respectively, indicating the classical confusion.

In [ ]:
channel = MeasurementChannel.from_readout_fidelity(inst=MEASURE(0, None), fidelity=0.95)
channel.plot()

### Measurement from confusion and transition matrices

We can also construct an instrument from the confusion and transition matrices.

The confusion matrix describes a classification error. Note that while confusion matrices are often written with the true states as the rows and the labels as the columns, here we have the true states and the columns and labels as the rows. Thus, the columns must sum to 1.

The confusion matrix describes **how well we can distinguish** computational basis states |0⟩ and |1⟩.

- **Definition**: $T_{i, j} = P(i | j) $
- **Convention**: Columns are prepared states, rows are measurement outcomes
- **Constraint**: Each columns sums to 1.0

The transition matrix describes post-measurement bitflip errors and is defined as $P(k | j)$ where $k$ is the final state and $j$ is the input state. So $P(1, 0)$ defines the probability of transition from 0 to 1.

- **Definition**: $T_{i, j} = P(|i⟩ | |j⟩) $
- **Convention**: Columns are input states, rows are output states  
- **Constraint**: Each column sums to 1.0
- **Interpretation**:
  - Diagonal elements (0,0) and (1,1): State preservation probabilities
  - Off-diagonal: State-flip probabilities
  - Non-QND fidelity = average of diagonal elements

Below we construct a quantum instrument with the confusion matrix

$M_{C} = \begin{bmatrix} 0.9 & 0.0 \\ 0.1 & 1.0 \end{bmatrix}$

and transition matrix

$M_{T} = \begin{bmatrix} 0.8 & 0.0 \\ 0.2 & 1.0 \end{bmatrix}$

We can see the effect of the transition matrix in the primary plot, as qubit has a 20% probabiltiy to excite from the 0 state to the 1 state after the measurement.

We can also see the effect of confusion in outcome 1, where input state 0 is sometimes classified as 1. We can see the effect of transition in the outcome 0 subplot, where the input state 0 is correctly classified, but sometimes transitions to 1. 


In [ ]:
channel = MeasurementChannel.from_confusion_and_transition(
    inst=MEASURE(0, None),
    confusion_matrix=jnp.array([[0.9, 0.0], [0.1, 1.0]]),
    transition_matrix=jnp.array([[0.8, 0.0], [0.2, 1.0]]),
)
channel.plot()

### 3-state measurement

We can also construct 3-state instruments. Here, we construct an instrument that has a 30% chance of promoting the 1 state to the 2 state (high leakage), but confuses the 2 state for 1 or 0 100% of the time with a strongly biased probability. This kind of measurement is a binary classifier applied to a leaky measurement.

In [ ]:
channel = MeasurementChannel.from_confusion_and_transition(
    inst=MEASURE(0, None),
    confusion_matrix=jnp.array([[1.0, 0.0, 0.1], [0.0, 1.0, 0.9], [0.0, 0.0, 0.0]]),
    transition_matrix=jnp.array([[1.0, 0.1, 0.1], [0.0, 0.6, 0.4], [0.0, 0.3, 0.5]]),
)
channel.plot()

### Applying measurements to states

Below we provide some examples of constructing and applying quantum instruments to different intial states

In [ ]:
#### Perfect QND Measurement
# Perfect classification, perfect state preservation
confusion = jnp.array([[1.0, 0.0], [0.0, 1.0]])
transition = jnp.array([[1.0, 0.0], [0.0, 1.0]])
instr = MeasurementChannel.from_confusion_and_transition(MEASURE(0, None), confusion, transition)
print(f"Perfect Instrument: {instr}")
# Result: fidelity=1.0, non_demolition_fidelity=1.0

#### Noisy but QND Measurement
# 95% classification accuracy, but state perfectly preserved
confusion = jnp.array([[0.95, 0.05], [0.05, 0.95]])
transition = jnp.array([[1.0, 0.0], [0.0, 1.0]])
instr = MeasurementChannel.from_confusion_and_transition(MEASURE(0, None), confusion, transition)
print(f"Noisy QND Instrument: {instr}")
# Result: fidelity=0.95, non_demolition_fidelity=1.0

#### Perfect Readout with Back-action
# Perfect classification, but measurement causes relaxation
confusion = jnp.array([[1.0, 0.0], [0.0, 1.0]])
transition = jnp.array([[0.9, 0.2], [0.1, 0.8]])
instr = MeasurementChannel.from_confusion_and_transition(MEASURE(0, None), confusion, transition)
print(f"Back-action Instrument: {instr}\n")
# Result: fidelity=1.0, non_demolition_fidelity=0.85

rho_zero = qx.zero_state_matrix(1)
rho_one = qx.gates.X @ rho_zero
rho_plus = qx.gates.RY(jnp.pi / 2) @ rho_zero
key = jax.random.key(374)

rho_in = rho_zero
print("Back-action Instrument: Input state: |0><0|")
rho_out, outcome = qx.apply_instrument_to_density_matrix(instr.process, rho_in, key)
print(f"  Outcome: {outcome}\n  Post-measurement state: {rho_out.pretty_print()}\n")

rho_in = rho_one
print("Back-action Instrument: Input state: |1><1|")
rho_out, outcome = qx.apply_instrument_to_density_matrix(instr.process, rho_in, key)
print(f"  Outcome: {outcome}\n  Post-measurement state: {rho_out.pretty_print()}\n")

rho_in = rho_plus
print("Back-action Instrument: Input state: |+><+|")
rho_out, outcome = qx.apply_instrument_to_density_matrix(instr.process, rho_in, key)
print(f"  Outcome: {outcome}\n  Post-measurement state: {rho_out.pretty_print()}\n")

## Calcuating the fidelity of circuits

Predicting the fidelity of a circuit from a noise model is an important task for the experimentalist.

Below, we demonstrates how we can predict the fidelity of a circuit from a given noise model. To do so, we make an important assumption that the noise is incoherent.

Another fact to note is that the multiplication of process fidelities fails for circuits with very low fidelities. For circuits pushing the limits of experimental feasibility, this may be an important fact.

### Construct the circuit and noise model

We will construct a noise model of depolarizing 99% ISWAP gates. The gates will be used in a layer circuit on a grid topology.

We'll need a few more imports for this.

## Further Reading

For advanced noise modeling features such as circuit fidelity estimation
and light cone analysis, see the `qpu-hybrid-benchmark` package.